- ulysess（尤利西斯，西方著名长篇小说）
    - DeepSpeed ulysess
- llama-factory 官方版本不支持序列并行
    - 360-LLaMA-Factory（社区 fork）

### basics

- ulysses_sequence_parallel_size=4 (序列并行)
    - 它将输入数据的 序列长度 (Sequence Length) 切分成 4 份。例如 16k 的输入，每张卡实际只计算 4k 长度的 Attention。
    - 解决单卡显存无法容纳超长序列 Activation 的问题
- sp of sft vs. rl
    - sft: 对 prompt + response 进行切分 (response 是监督数据)
    - rl: inference engine 完成 response 的 rollout，然后仅需 sp

- Padding & Slicing:
    - SFT: verl/trainer/fsdp_sft_trainer.py 调用 ulysses_pad_and_slice_inputs
    - RL: verl/workers/actor/dp_actor.py 调用 ulysses_pad_and_slice_inputs
    - 先 padding 到能被 sp_size 整除的长度，然后 slice_input_tensor 进行切分。
- Attention 计算:
    - 都依赖于 Monkey Patch 后的 _ulysses_flash_attention_forward (在 verl/models/transformers/monkey_patch.py)，通过 All-to-All 通信完成注意力计算。

### verl

```
self.ulysses_sequence_parallel_size = self.config.ulysses_sequence_parallel_size
self.use_ulysses_sp = self.ulysses_sequence_parallel_size > 1
```
```python
actor_rollout_ref.actor.ppo_max_token_len_per_gpu=8192 \
actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=8192 \
actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=8192 \
```